# Retrieval-Augmented Generation (RAG)

## Learning Objectives
1. Understand how retrievers and generators work together in RAG pipelines
2. Implement a basic dense retriever using sentence embeddings
3. Build an end-to-end RAG system for open-domain question-answering
4. Analyze the impact of retriever quality, chunk size, and K on end-to-end accuracy

## Cell 2: Imports and Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from transformers import AutoTokenizer, AutoModel
try:
    from sentence_transformers import SentenceTransformer, util
except ImportError:
    print("Installing sentence-transformers...")
    import subprocess
    subprocess.check_call(["pip", "install", "sentence-transformers", "-q"])
    from sentence_transformers import SentenceTransformer, util

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

## Level 1: Basic Dense Retriever (Cosine Similarity)

A dense retriever encodes both documents and queries into a shared embedding space.
We retrieve the most similar documents using cosine similarity.

In [ ]:
# Simple knowledge base: facts about countries and cities
documents = [
    "Paris is the capital of France, known for the Eiffel Tower and museums.",
    "The Eiffel Tower is 330 meters tall and was built in 1889.",
    "France has a population of about 67 million people.",
    "Tokyo is the capital and largest city of Japan.",
    "Japan is an island nation in East Asia with a population of 125 million.",
    "Mount Fuji is the highest mountain in Japan at 3,776 meters.",
    "Berlin is the capital of Germany, known for its history and culture.",
    "Germany is located in Central Europe and has a population of 83 million.",
]

class SimpleRetriever:
    """Encodes documents and queries into embeddings, retrieves by cosine similarity."""
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.documents = []
        self.doc_embeddings = None

    def index_documents(self, docs):
        """Encode and store all documents."""
        self.documents = docs
        self.doc_embeddings = self.model.encode(docs, convert_to_tensor=True)
        print(f"Indexed {len(docs)} documents, embedding dimension: {self.doc_embeddings.shape[1]}")

    def retrieve(self, query, top_k=3):
        """Retrieve top-K documents for a query."""
        q_embedding = self.model.encode(query, convert_to_tensor=True)
        # Cosine similarity
        scores = util.pytorch_cos_sim(q_embedding, self.doc_embeddings)[0]
        top_k_idx = scores.topk(top_k).indices
        top_k_docs = [self.documents[i] for i in top_k_idx.cpu().numpy()]
        top_k_scores = [scores[i].item() for i in top_k_idx]
        return top_k_docs, top_k_scores

# Build retriever
retriever = SimpleRetriever(model_name="all-MiniLM-L6-v2")
retriever.index_documents(documents)

# Test queries
test_queries = [
    "What is the capital of France?",
    "How tall is Mount Fuji?",
    "Which city is the capital of Germany?",
]

print("\n=== Retrieval Results ===")
for query in test_queries:
    docs, scores = retriever.retrieve(query, top_k=3)
    print(f"\nQuery: {query}")
    for i, (doc, score) in enumerate(zip(docs, scores), 1):
        print(f"  [{i}] (score={score:.3f}) {doc[:60]}...")

## Level 2: End-to-End RAG with Learned Retriever

Build a learnable retriever that can be fine-tuned on a QA task.
Encode questions and documents separately, then rank by relevance.
Train to maximize ranking of correct documents.

In [ ]:
class LearnableRetriever(nn.Module):
    """Encoder-based retriever: learns to match questions and documents."""
    def __init__(self, model_name="bert-base-uncased", embedding_dim=256):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.hidden_dim = self.encoder.config.hidden_size
        self.embedding_dim = embedding_dim
        # Project from BERT hidden to embedding space
        self.proj = nn.Linear(self.hidden_dim, embedding_dim)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    def encode(self, texts, max_length=128):
        """Encode texts to embeddings (mean pooling + projection)."""
        with torch.no_grad():
            encoded = self.tokenizer(texts, padding=True, truncation=True,
                                     max_length=max_length, return_tensors="pt")
            encoded = {k: v.to(device) for k, v in encoded.items()}
            outputs = self.encoder(**encoded)
            # Mean pooling
            mask = encoded["attention_mask"].unsqueeze(-1)
            embeddings = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1)
            # Project and normalize
            embeddings = self.proj(embeddings)
            embeddings = F.normalize(embeddings, dim=-1)
        return embeddings

    def forward(self, questions, documents):
        """Score: question-document similarity."""
        with torch.no_grad():
            q_emb = self.encode(questions)  # (B, D)
            doc_emb = self.encode(documents)  # (N, D)
        # Scores: (B, N) matrix of similarities
        scores = torch.mm(q_emb, doc_emb.t())  # inner product (cosine after L2 norm)
        return scores, q_emb, doc_emb

# Synthetic QA dataset
qa_pairs = [
    ("What is the capital of France?", "Paris is the capital of France, known for the Eiffel Tower and museums."),
    ("How tall is the Eiffel Tower?", "The Eiffel Tower is 330 meters tall and was built in 1889."),
    ("What is the population of France?", "France has a population of about 67 million people."),
    ("What is the capital of Japan?", "Tokyo is the capital and largest city of Japan."),
    ("How many people live in Japan?", "Japan is an island nation in East Asia with a population of 125 million."),
    ("What is the height of Mount Fuji?", "Mount Fuji is the highest mountain in Japan at 3,776 meters."),
    ("Where is Berlin?", "Berlin is the capital of Germany, known for its history and culture."),
    ("What is the population of Germany?", "Germany is located in Central Europe and has a population of 83 million."),
]

# Prepare training: questions, positive docs (correct), negative docs (incorrect)
questions = [q for q, d in qa_pairs]
positive_docs = [d for q, d in qa_pairs]
negative_docs = [qa_pairs[(i + 1) % len(qa_pairs)][1] for i in range(len(qa_pairs))]
all_docs = list(set(positive_docs + negative_docs))

print(f"QA Pairs: {len(qa_pairs)}, Unique docs: {len(all_docs)}")
print("Sample QA pair:")
print(f"  Q: {qa_pairs[0][0]}")
print(f"  Pos doc: {qa_pairs[0][1]}")

## Real-World Example 1: RAG for Open-Domain QA

Combine retriever and simple generator.
For each question, retrieve top-K documents and return a template answer.
Measure accuracy: does the retrieved set contain the correct answer document?

In [ ]:
def template_generator(question, retrieved_docs):
    """Simple template-based answer generation (stand-in for neural generator).
    
    In practice, this would be a seq2seq model reading question + retrieved docs.
    Here we simulate by returning the first retrieved doc as the answer.
    """
    if retrieved_docs:
        return retrieved_docs[0]  # Return first retrieved doc as answer
    return "No documents found."

def rag_pipeline(retriever, question, top_k=3, verbose=False):
    """Full RAG pipeline: retrieve then generate."""
    # Retrieve
    retrieved_docs, scores = retriever.retrieve(question, top_k=top_k)
    # Generate (here: simple template)
    answer = template_generator(question, retrieved_docs)
    return {
        "question": question,
        "retrieved_docs": retrieved_docs,
        "retrieval_scores": scores,
        "answer": answer,
    }

# Test RAG on all queries
print("\n=== RAG Pipeline Results ===")
for question in test_queries:
    result = rag_pipeline(retriever, question, top_k=3)
    print(f"\nQ: {question}")
    print(f"A: {result['answer'][:80]}...")
    print(f"Retrieved {len(result['retrieved_docs'])} docs (scores: {[f'{s:.2f}' for s in result['retrieval_scores']]})") 

## Real-World Example 2: Impact of K and Chunk Strategy on Accuracy

Evaluate how retrieval quality varies with different top-K values and document chunking.
Measure: retrieval recall at K (did the correct document make it into top-K?)

In [ ]:
def evaluate_retriever(retriever, qa_pairs, top_k_values=[1, 3, 5, 10]):
    """Evaluate retriever by measuring recall at K."""
    results = {k: [] for k in top_k_values}
    
    for question, correct_doc in qa_pairs:
        retrieved_docs, scores = retriever.retrieve(question, top_k=max(top_k_values))
        for k in top_k_values:
            # Check if correct doc is in top-K
            found = any(correct_doc == doc for doc in retrieved_docs[:k])
            results[k].append(1.0 if found else 0.0)
    
    return {k: np.mean(v) for k, v in results.items()}

# Evaluate baseline retriever
recall_results = evaluate_retriever(retriever, qa_pairs, top_k_values=[1, 2, 3, 5])
print("\n=== Retriever Recall@K ===")
for k, recall in recall_results.items():
    print(f"Recall@{k}: {recall:.4f} ({int(recall * len(qa_pairs))}/{len(qa_pairs)} correct)")

# Simulate chunking impact: split docs into smaller chunks
def chunk_documents(docs, chunk_size=50):
    """Split documents into chunks of approximately chunk_size tokens."""
    chunked = []
    for doc in docs:
        words = doc.split()
        # Split into chunks of roughly chunk_size words
        for i in range(0, len(words), chunk_size):
            chunk = " ".join(words[i:i+chunk_size])
            if chunk.strip():
                chunked.append(chunk)
    return chunked

# Test with different chunk sizes
chunk_sizes = [50, 100, 200]
chunk_results = {}

for chunk_size in chunk_sizes:
    chunked_docs = chunk_documents(documents, chunk_size=chunk_size)
    chunked_retriever = SimpleRetriever()
    chunked_retriever.index_documents(chunked_docs)
    
    # Map chunked docs back to QA pairs for evaluation
    # (simplified: check if any chunk contains answer)
    recall_k3 = []
    for question, correct_doc in qa_pairs:
        retrieved_chunks, _ = chunked_retriever.retrieve(question, top_k=3)
        # Check if any chunk contains key words from correct_doc
        found = any(correct_doc[:30] in chunk for chunk in retrieved_chunks)
        recall_k3.append(1.0 if found else 0.0)
    
    chunk_results[chunk_size] = np.mean(recall_k3)

print("\n=== Impact of Chunking Strategy (Recall@3) ===")
for chunk_size, recall in chunk_results.items():
    print(f"Chunk size {chunk_size:3d} words: {recall:.4f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# Recall@K curve
ks = list(recall_results.keys())
recalls = [recall_results[k] for k in ks]
axes[0].plot(ks, recalls, marker='o', linewidth=2, markersize=8, color='steelblue')
axes[0].set_xlabel("K (top-K retrieved)")
axes[0].set_ylabel("Recall")
axes[0].set_title("Retrieval Recall@K")
axes[0].grid(alpha=0.3)
axes[0].set_xticks(ks)

# Chunk size impact
chunk_szs = list(chunk_results.keys())
chunk_recalls = [chunk_results[s] for s in chunk_szs]
axes[1].bar(range(len(chunk_szs)), chunk_recalls, color='coral', alpha=0.7,
            tick_label=[f"{s}" for s in chunk_szs])
axes[1].set_xlabel("Chunk Size (words)")
axes[1].set_ylabel("Recall@3")
axes[1].set_title("Chunking Strategy Impact")
axes[1].set_ylim([0, 1.1])

plt.tight_layout()
plt.savefig("/tmp/rag_retrieval_analysis.png", dpi=80, bbox_inches='tight')
plt.show()
print("\nPlot saved to /tmp/rag_retrieval_analysis.png")

## Real-World Example 3: Retriever-Generator Interaction

Simulate how generator performance depends on retrieved passages.
Measure: can the generator answer the question given the retrieved docs?

In [ ]:
def simulate_generator_accuracy(retrieved_docs, correct_answer, threshold=0.6):
    """Simulate generator: returns correct answer if retrieved docs contain key info.
    
    In practice, a neural generator reads all docs and generates tokens.
    Here we simulate by checking if correct answer substring appears in retrieved docs.
    """
    if not retrieved_docs:
        return False
    
    # Extract key phrases from correct answer
    key_terms = correct_answer.split()[:3]  # First 3 words
    context = " ".join(retrieved_docs).lower()
    
    # Check how many key terms appear in retrieved context
    matches = sum(1 for term in key_terms if term.lower() in context)
    return matches >= len(key_terms) * threshold

def end_to_end_evaluation(retriever, qa_pairs, top_k_values=[1, 3, 5]):
    """Evaluate full RAG system: retriever + generator."""
    results = {k: {'retriever_recall': [], 'generator_accuracy': []} for k in top_k_values}
    
    for question, correct_doc in qa_pairs:
        retrieved_docs, scores = retriever.retrieve(question, top_k=max(top_k_values))
        
        for k in top_k_values:
            retrieved_k = retrieved_docs[:k]
            
            # Retriever metric: is correct doc in top-K?
            retriever_hit = any(correct_doc == doc for doc in retrieved_k)
            results[k]['retriever_recall'].append(1.0 if retriever_hit else 0.0)
            
            # Generator metric: can it answer given retrieved docs?
            gen_success = simulate_generator_accuracy(retrieved_k, correct_doc)
            results[k]['generator_accuracy'].append(1.0 if gen_success else 0.0)
    
    # Average over all QA pairs
    return {k: {m: np.mean(v) for m, v in metrics.items()} 
            for k, metrics in results.items()}

# Evaluate full pipeline
e2e_results = end_to_end_evaluation(retriever, qa_pairs, top_k_values=[1, 2, 3, 5])

print("\n=== End-to-End RAG Evaluation ===")
print(f"{'K':>3} | {'Retriever Recall':>16} | {'Generator Accuracy':>17} | {'End-to-End':>10}")
print("-" * 60)
for k in sorted(e2e_results.keys()):
    ret_rec = e2e_results[k]['retriever_recall']
    gen_acc = e2e_results[k]['generator_accuracy']
    e2e = min(ret_rec, gen_acc)  # Pessimistic: both must succeed
    print(f"{k:>3} | {ret_rec:>16.4f} | {gen_acc:>17.4f} | {e2e:>10.4f}")

# Key insight: bottleneck analysis
print("\n=== Bottleneck Analysis ===")
k_best = 3
ret_recall = e2e_results[k_best]['retriever_recall']
gen_acc = e2e_results[k_best]['generator_accuracy']
if ret_recall < gen_acc:
    print(f"Bottleneck at K={k_best}: RETRIEVER ({ret_recall:.2%}) < Generator ({gen_acc:.2%})")
    print("  → Solution: improve retriever (better encoder, more training data, re-ranking)")
elif gen_acc < ret_recall:
    print(f"Bottleneck at K={k_best}: GENERATOR ({gen_acc:.2%}) < Retriever ({ret_recall:.2%})")
    print("  → Solution: improve generator (better model, more context, filtering contradictions)")
else:
    print(f"Balanced: Retriever and Generator both at {ret_recall:.2%}")

## Comparison: Retriever Types and Trade-offs

Compare dense (semantic) vs keyword (lexical) retrieval.
Show speed vs accuracy trade-offs.

In [ ]:
import time

def bm25_retrieve(query, docs, top_k=3):
    """Simulate BM25 (keyword-based) retrieval.
    Simple version: count overlapping tokens.
    """
    query_words = set(query.lower().split())
    scores = []
    for doc in docs:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words)
        scores.append(overlap)
    
    top_k_idx = np.argsort(scores)[-top_k:][::-1]
    return [docs[i] for i in top_k_idx], [scores[i] for i in top_k_idx]

# Compare retrievers
retriever_types = {
    'Dense (SBERT)': lambda q: retriever.retrieve(q, top_k=3),
    'Keyword (BM25)': lambda q: bm25_retrieve(q, documents, top_k=3),
}

print("\n=== Retriever Comparison ===")
for query in test_queries[:2]:
    print(f"\nQuery: {query}")
    for ret_type, ret_func in retriever_types.items():
        start = time.time()
        docs, scores = ret_func(query)
        elapsed = (time.time() - start) * 1000  # ms
        print(f"  {ret_type}:")
        for doc, score in zip(docs, scores):
            print(f"    (score={score:.2f}) {doc[:50]}...")
        print(f"    Time: {elapsed:.2f} ms")

# Measure accuracy and speed trade-off
metrics_by_type = {}
for ret_name, ret_func in retriever_types.items():
    hits = 0
    times = []
    for question, correct_doc in qa_pairs:
        start = time.time()
        docs, _ = ret_func(question)
        elapsed = time.time() - start
        times.append(elapsed * 1000)
        if any(correct_doc == doc for doc in docs):
            hits += 1
    
    metrics_by_type[ret_name] = {
        'accuracy': hits / len(qa_pairs),
        'latency_ms': np.mean(times),
    }

print("\n=== Speed vs Accuracy Comparison ===")
print(f"{'Retriever':>20} | {'Accuracy':>10} | {'Latency (ms)':>12}")
print("-" * 48)
for ret_type, metrics in metrics_by_type.items():
    print(f"{ret_type:>20} | {metrics['accuracy']:>10.4f} | {metrics['latency_ms']:>12.2f}")

# Visualize trade-off
fig, ax = plt.subplots(figsize=(8, 5))
for ret_type, metrics in metrics_by_type.items():
    ax.scatter(metrics['latency_ms'], metrics['accuracy'], s=300, alpha=0.7, label=ret_type)
ax.set_xlabel("Latency (ms)")
ax.set_ylabel("Accuracy (Recall@3)")
ax.set_title("Retriever: Speed vs Accuracy Trade-off")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/rag_tradeoff.png", dpi=80, bbox_inches='tight')
plt.show()
print("\nPlot saved to /tmp/rag_tradeoff.png")

## Key Takeaways

**Core idea:**
RAG separates retrieval (fast, approximate search) from generation (careful reasoning).
This modularity enables scaling to millions of documents and transparent, updatable knowledge.

**Key trade-offs:**
- **K (top-K passages):** Larger K improves accuracy but increases latency and token consumption. K=3–5 is typical.
- **Chunk size:** Smaller chunks (100–200 tokens) improve retrieval precision but increase index size. Larger chunks (500+ tokens) save space but may miss specific info.
- **Retriever type:** Dense embeddings are more accurate for semantic tasks; BM25 is faster and better for keyword-based queries. Hybrid (both) is robust.
- **Training:** Pre-trained + fine-tuned retriever >> pre-trained only. End-to-end training >> separate training.

**Bottleneck analysis:**
Measure retriever recall and generator accuracy separately. Usually retriever is the bottleneck early on; once it hits ~90%, focus on generator quality.

**Common pitfalls:**
1. Retriever-generator mismatch: pre-trained on different domains
2. Poor chunking: answers split across chunks
3. Stale index: documents added but index not updated
4. Too few passages: K=1 is risky; use K≥3

**Related concepts:**
- Dense retriever pre-training: DPR, ColBERT, BGE
- Generator improvements: FLAN, Llama, GPT with in-context learning
- Vector databases: FAISS, Qdrant, Weaviate for scaling